# 24. Detecting Hallucinations & Model Drift
**Duration:** 25 min | **Topics:** Quality assurance, drift detection

## Overview
In this notebook, you'll learn to:
- Detect **LLM hallucinations** using factual consistency checks
- Implement **model drift detection** over time
- Build an automated **quality assurance pipeline**
- Use **Azure ML Data Drift** monitoring for LLM outputs
- Score responses with **reference-free evaluation** metrics

### Minimum Azure Resources Required
| Resource | SKU | Est. Cost |
|---|---|---|
| Azure ML Workspace | Basic | Free (compute is extra) |
| Azure ML Compute | Standard_DS2_v2 | ~$0.10/hr |
| Application Insights | Pay-as-you-go | 5 GB free/month |

In [ ]:
import subprocess, sys, os

def safe_install(packages):
    """
    Install packages safely in Azure ML environments.
    Suppresses ALL known pre-existing base-image conflicts:
      - azureml-automl-*, azureml-datadrift, azureml-interpret, azureml-train-*
      - torch version mismatch (2.9.x vs required 2.2.2)
      - numpy version mismatch (1.26.x / 2.x vs <=1.23.5)
      - psutil version mismatch (7.x vs <5.9.4)
      - matplotlib version mismatch (3.7.x vs <=3.6.3)
      - pandas-ml missing enum34
      - jupyterlab-nvdashboard jupyterlab version
    None of these affect notebook functionality.
    """
    KNOWN_CONFLICTS = [
        'azureml', 'nvdashboard', 'pandas-ml', 'enum34',
        'torch==', 'numpy<=', 'numpy>=', 'psutil<', 'psutil>=',
        'matplotlib<=', 'matplotlib>=',
    ]

    # Pass 1: install without dep resolver (fast, no conflict noise)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check', '--no-deps', *packages],
        capture_output=True
    )

    # Pass 2: normal install to pull in genuine transitive deps
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check', *packages],
        capture_output=True, text=True
    )

    # Only surface errors that are NOT known Azure ML base-image conflicts
    real_errors = [
        line for line in (result.stderr or '').splitlines()
        if 'ERROR' in line
        and not any(k in line for k in KNOWN_CONFLICTS)
    ]

    if real_errors:
        print('⚠️  Unexpected install error (please report):')
        for e in real_errors:
            print(f'   {e}')
    else:
        print(f'✅ Ready: {", ".join(packages)}')
        print('   (Pre-existing Azure ML env warnings suppressed — safe to ignore)')

safe_install(['sentence-transformers', 'scikit-learn', 'nltk', 'rouge-score'])


## Step 1: Hallucination Detection Methods

In [ ]:
import re
import json
import numpy as np
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

@dataclass
class EvaluationResult:
    method: str
    score: float  # 0.0 (bad) to 1.0 (good)
    label: str    # "hallucination", "ok", "uncertain"
    details: dict

    def __repr__(self):
        emoji = "✅" if self.label == "ok" else ("⚠️" if self.label == "uncertain" else "🚨")
        return f"{emoji} [{self.method}] score={self.score:.3f} → {self.label}"


class HallucinationDetector:
    """Multi-strategy hallucination detection for LLM outputs."""

    def __init__(self):
        # Known facts database (in production: use Azure AI Search)
        self.fact_db = {
            "azure openai gpt-4o release": "2024",
            "azure ml workspace pricing": "free basic tier",
            "azure container apps min replicas": "0",
            "azure functions free tier requests": "1 million",
        }

        # Uncertainty signals
        self.uncertainty_phrases = [
            "i think", "i believe", "probably", "might be", "could be",
            "i'm not sure", "approximately", "around", "roughly"
        ]

        # Confident hallucination signals
        self.hallucination_signals = [
            r"as of \d{4}",  # outdated date references
            r"\$\d+\.\d{2}",  # specific prices that may be wrong
        ]

    def check_factual_consistency(self, response: str, context: Optional[str] = None) -> EvaluationResult:
        """Check if response contradicts known facts."""
        response_lower = response.lower()
        contradictions = []

        for fact_key, fact_value in self.fact_db.items():
            if any(word in response_lower for word in fact_key.split()):
                if fact_value.lower() not in response_lower:
                    contradictions.append(f"Possible contradiction with: {fact_key}")

        score = max(0.0, 1.0 - (len(contradictions) * 0.3))
        label = "ok" if score > 0.7 else ("uncertain" if score > 0.4 else "hallucination")

        return EvaluationResult(
            method="factual_consistency",
            score=score,
            label=label,
            details={"contradictions": contradictions}
        )

    def check_uncertainty_calibration(self, response: str) -> EvaluationResult:
        """Check if the model appropriately signals uncertainty."""
        response_lower = response.lower()
        uncertainty_count = sum(1 for p in self.uncertainty_phrases if p in response_lower)
        total_sentences = max(1, len(response.split(".")))

        # Penalize overconfident responses on uncertain topics
        hallucination_pattern_count = sum(
            1 for p in self.hallucination_signals if re.search(p, response)
        )

        # More uncertainty phrases = better calibrated
        calibration_score = min(1.0, 0.5 + (uncertainty_count / total_sentences))
        if hallucination_pattern_count > 0:
            calibration_score = max(0.1, calibration_score - 0.3)

        label = "ok" if calibration_score > 0.6 else "uncertain"

        return EvaluationResult(
            method="uncertainty_calibration",
            score=calibration_score,
            label=label,
            details={"uncertainty_phrases": uncertainty_count,
                     "hallucination_patterns": hallucination_pattern_count}
        )

    def check_length_anomaly(self, response: str, expected_min: int = 50, expected_max: int = 2000) -> EvaluationResult:
        """Flag responses that are suspiciously short or extremely long."""
        length = len(response)
        if expected_min <= length <= expected_max:
            score, label = 1.0, "ok"
        elif length < expected_min:
            score = max(0.1, length / expected_min)
            label = "hallucination"  # truncated/empty = likely error
        else:
            score = max(0.3, 1.0 - (length - expected_max) / expected_max)
            label = "uncertain"

        return EvaluationResult(
            method="length_anomaly",
            score=score,
            label=label,
            details={"length": length, "expected_range": (expected_min, expected_max)}
        )

    def evaluate(self, response: str, context: Optional[str] = None) -> Dict:
        """Run all checks and return aggregated hallucination score."""
        results = [
            self.check_factual_consistency(response, context),
            self.check_uncertainty_calibration(response),
            self.check_length_anomaly(response),
        ]
        avg_score = np.mean([r.score for r in results])
        has_hallucination = any(r.label == "hallucination" for r in results)
        final_label = "hallucination" if has_hallucination else ("ok" if avg_score > 0.6 else "uncertain")

        return {
            "overall_score": round(avg_score, 3),
            "label": final_label,
            "checks": [str(r) for r in results],
            "details": {r.method: r.details for r in results}
        }


detector = HallucinationDetector()

# Test with various responses
test_responses = [
    ("Good response",
     "Azure Container Apps supports scaling to 0 replicas, which is great for cost savings. I believe the pricing is consumption-based."),
    ("Possible hallucination",
     "Azure Functions offers 2 million free requests per month as of 2023. The cost is exactly $0.00012 per additional request."),
    ("Too short",
     "Yes."),
]

print("🔍 Hallucination Detection Results")
print("=" * 60)
for name, response in test_responses:
    result = detector.evaluate(response)
    print(f"\n📝 [{name}]")
    print(f"   Text: {response[:100]}..." if len(response) > 100 else f"   Text: {response}")
    print(f"   Overall Score: {result['overall_score']} → {result['label'].upper()}")
    for check in result['checks']:
        print(f"   {check}")

## Step 2: Model Drift Detection Over Time

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

class ModelDriftDetector:
    """Detect statistical drift in LLM response quality over time."""

    def __init__(self, window_size: int = 100):
        self.window_size = window_size
        self.baseline_scores = []
        self.current_scores = []

    def set_baseline(self, scores: List[float]):
        """Set baseline distribution from known-good period."""
        self.baseline_scores = scores
        print(f"✅ Baseline set: n={len(scores)}, mean={np.mean(scores):.3f}, std={np.std(scores):.3f}")

    def add_scores(self, scores: List[float]):
        self.current_scores.extend(scores)
        self.current_scores = self.current_scores[-self.window_size:]

    def detect_drift(self) -> Dict:
        """KS-test to detect distribution shift."""
        if len(self.baseline_scores) < 10 or len(self.current_scores) < 10:
            return {"drift_detected": False, "reason": "Insufficient data"}

        ks_stat, p_value = stats.ks_2samp(self.baseline_scores, self.current_scores)
        drift_detected = bool(p_value < 0.05)  # 95% confidence — cast to Python bool for JSON

        baseline_mean = np.mean(self.baseline_scores)
        current_mean = np.mean(self.current_scores)
        relative_change = (current_mean - baseline_mean) / baseline_mean * 100

        return {
            "drift_detected":      drift_detected,
            "ks_statistic":        float(round(ks_stat, 4)),
            "p_value":             float(round(p_value, 4)),
            "baseline_mean":       float(round(baseline_mean, 3)),
            "current_mean":        float(round(current_mean, 3)),
            "relative_change_pct": float(round(relative_change, 1)),
            "severity":            "high" if abs(relative_change) > 10 else ("medium" if abs(relative_change) > 5 else "low"),
        }


# Simulate drift scenario
np.random.seed(42)
drift_detector = ModelDriftDetector(window_size=100)

# Baseline: week 1-2 scores
baseline = np.random.beta(8, 2, 200).tolist()  # High quality: mean ~0.8
drift_detector.set_baseline(baseline)

# Simulate drift: week 3-4 (quality drops after model update)
drifted = np.random.beta(5, 4, 200).tolist()  # Lower quality: mean ~0.55
drift_detector.add_scores(drifted)

result = drift_detector.detect_drift()
emoji = "🚨" if result["drift_detected"] else "✅"
print(f"\n{emoji} Drift Detection Result:")
print(json.dumps(result, indent=2))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model Drift Detection', fontsize=14, fontweight='bold')

axes[0].hist(baseline[:200], bins=30, alpha=0.6, label='Baseline', color='#0078D4')
axes[0].hist(drifted[:200], bins=30, alpha=0.6, label='Current (drifted)', color='#D83B01')
axes[0].set_title('Score Distribution Shift')
axes[0].legend()
axes[0].set_xlabel('Quality Score')
axes[0].grid(True, alpha=0.3)

# Rolling mean over time
all_scores = baseline[:100] + drifted[:100]
rolling_mean = [np.mean(all_scores[max(0, i-20):i+1]) for i in range(len(all_scores))]
axes[1].plot(range(100), rolling_mean[:100], color='#0078D4', label='Baseline period', linewidth=2)
axes[1].plot(range(100, 200), rolling_mean[100:], color='#D83B01', label='Drift period', linewidth=2)
axes[1].axvline(x=100, color='orange', linestyle='--', label='Model update')
axes[1].set_title('Rolling Mean Quality Score')
axes[1].legend()
axes[1].set_xlabel('Request #')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_drift.png', dpi=100, bbox_inches='tight')
plt.close()
print("\n✅ Drift visualization saved as model_drift.png")

## Step 3: Automated QA Pipeline

In [ ]:
class LLMQualityPipeline:
    """End-to-end quality assurance pipeline for LLM outputs."""

    def __init__(self, hallucination_threshold: float = 0.6, drift_alert_threshold: float = -10.0):
        self.hallucination_detector = HallucinationDetector()
        self.drift_detector = ModelDriftDetector()
        self.hallucination_threshold = hallucination_threshold
        self.drift_alert_threshold = drift_alert_threshold
        self.scores_history = []

    def evaluate_response(self, response: str, context: Optional[str] = None) -> Dict:
        """Evaluate a single LLM response."""
        result = self.hallucination_detector.evaluate(response, context)
        self.scores_history.append(result["overall_score"])

        # Auto-alert
        alert = None
        if result["overall_score"] < self.hallucination_threshold:
            alert = f"🚨 LOW QUALITY RESPONSE DETECTED (score={result['overall_score']})"

        return {**result, "alert": alert}

    def run_batch_evaluation(self, test_cases: List[Dict]) -> Dict:
        """Evaluate a batch of responses and return summary."""
        results = []
        for case in test_cases:
            r = self.evaluate_response(case["response"], case.get("context"))
            r["test_name"] = case["name"]
            results.append(r)

        scores = [r["overall_score"] for r in results]
        pass_rate = sum(1 for s in scores if s >= self.hallucination_threshold) / len(scores)

        return {
            "total": len(results),
            "pass_rate": round(pass_rate * 100, 1),
            "mean_score": round(np.mean(scores), 3),
            "min_score": round(min(scores), 3),
            "results": results
        }


# Run batch QA
pipeline = LLMQualityPipeline()

test_suite = [
    {"name": "Accurate Azure info",
     "response": "Azure Container Apps can scale to zero replicas saving costs. I believe the consumption plan pricing is very affordable for low-traffic workloads."},
    {"name": "Hallucinated pricing",
     "response": "Azure charges exactly $1.23 per container app per hour as of 2023. This pricing is fixed and guaranteed."},
    {"name": "Empty response",
     "response": "OK."},
    {"name": "Well-calibrated uncertainty",
     "response": "Azure ML approximately supports around 50+ compute SKUs, though I'm not sure of the exact current number. You should check the Azure pricing page for the most up-to-date information."},
]

batch_result = pipeline.run_batch_evaluation(test_suite)

print("\n📊 Batch QA Results")
print("=" * 60)
print(f"  Total: {batch_result['total']} | Pass Rate: {batch_result['pass_rate']}% | Mean Score: {batch_result['mean_score']}")
for r in batch_result["results"]:
    status = "✅" if r["overall_score"] >= 0.6 else "🚨"
    print(f"  {status} {r['test_name']}: {r['overall_score']} ({r['label']})")

print("\n\n📌 Key Takeaways:")
print("  • No single method catches all hallucinations — use multiple checks")
print("  • KS-test detects distribution drift after model updates")
print("  • Log quality scores to App Insights for trending")
print("  • Set pass-rate thresholds in your CI/CD pipeline")

In [ ]:
# Azure ML Data Drift Monitor Setup
# Run these commands in Azure Cloud Shell after registering your datasets.

RG = 'rg-llm-demo'
WS = 'ws-llm-demo'

steps = [
    ('1. Register baseline dataset',
     f'az ml data create --name llm-quality-baseline --path ./baseline_scores.csv --type uri_file --workspace-name {WS} --resource-group {RG}'),
    ('2. Register production target dataset',
     f'az ml data create --name llm-quality-production --path azureml://datastores/workspaceblobstore/paths/quality/ --type mltable --workspace-name {WS} --resource-group {RG}'),
    ('3. Create daily drift monitor schedule',
     f'az ml schedule create --file drift_monitor.yaml --workspace-name {WS} --resource-group {RG}'),
    ('4. View drift results in Studio',
     f'az ml job list --workspace-name {WS} --resource-group {RG}'),
]

print('=' * 70)
print('  Azure ML Data Drift Monitor Setup')
print('=' * 70)
for title, cmd in steps:
    print(f'\n--- {title} ---')
    print(cmd)

print('\ndrift_monitor.yaml (schedule + KS-test runner):')
print('  name: llm-quality-drift-monitor')
print('  schedule: type=recurrence frequency=day interval=1')
print('  command: python drift_check.py --baseline llm-quality-baseline --threshold 0.05')

print('\n[SYNTHETIC DEMO OUTPUT]')
print('Schedule llm-quality-drift-monitor created (daily at 00:00 UTC)')
print('Day 1:  p=0.412  drift=False  mean_delta=-1.2%  OK')
print('Day 15: p=0.003  drift=True   mean_delta=-31.5% ALERT sent -> rollback applied')
print('Day 17: p=0.381  drift=False  mean_delta=-2.1%  OK  recovered')

print('\nKey Takeaways:')
print('  * Run ALL three hallucination checks — each catches different failure modes')
print('  * ROUGE for reference-based eval (RAG), hallucination detector for open-ended')
print('  * KS-test p < 0.05 = statistically significant drift — investigate immediately')
print('  * CI/CD quality gate: block deployment if pass-rate < 80%')
print('  * Human feedback thumbs-up/down builds labelled dataset for next eval cycle')
print('  * Azure ML drift monitor automates daily statistical testing in production')
